In [ ]:
import pandas as pd
import numpy as np

In [ ]:
breaks = pd.read_csv("data/raw/Water_Main_Breaks.csv")
mains = pd.read_csv("data/raw/Water_Mains.csv")

In [ ]:
print(breaks.shape)
breaks.sample(3)

In [ ]:
print(mains.shape)
mains.sample(3)

In [ ]:
breaks.columns

In [ ]:
mains.columns

In [ ]:
merged_data = mains.merge(breaks, left_on='WATMAINID', right_on='ASSETID', how='outer')

In [ ]:
merged_data['X'] = merged_data['X'].fillna(method='ffill').fillna(method='bfill')
merged_data['Y'] = merged_data['Y'].fillna(method='ffill').fillna(method='bfill')

In [ ]:
merged_data.sample(4)

In [ ]:
merged_data.info()

In [ ]:
merged_data.columns

In [ ]:
# show info for X and Y columns
merged_data[['X', 'Y']].info()

In [ ]:
merged_data.shape

In [ ]:
# move the X and Y columns to the front of the dataframe
column_order = merged_data.columns.tolist()
column_order.insert(0, column_order.pop(column_order.index('X')))
column_order.insert(1, column_order.pop(column_order.index('Y')))
merged_data = merged_data.reindex(columns=column_order)

In [ ]:
merged_data.sample(4)

In [ ]:
grouped_data = merged_data.groupby('WATMAINID')

In [ ]:
filled_data = grouped_data.apply(lambda group: group.fillna(method='ffill').fillna(method='bfill'))

In [ ]:
filled_data.reset_index(drop=True, inplace=True)

In [ ]:
missing_values_before = merged_data.isnull().sum()
missing_values_after = filled_data.isnull().sum()

print("Missing values before filling:")
print(missing_values_before)
print("\nMissing values after filling:")
print(missing_values_after)

In [ ]:
import matplotlib.pyplot as plt

# Choose a numeric column for comparison
column_name = 'PIPE_SIZE'

# Plot histograms before and after filling
plt.figure(figsize=(12, 6))
plt.subplot(121)
plt.hist(merged_data[column_name].dropna(), bins=30)
plt.title(f'Histogram of {column_name} (before filling)')

plt.subplot(122)
plt.hist(filled_data[column_name].dropna(), bins=30)
plt.title(f'Histogram of {column_name} (after filling)')

plt.show()

# Compare descriptive statistics
print("Descriptive statistics (before filling):")
print(merged_data[column_name].describe())
print("\nDescriptive statistics (after filling):")
print(filled_data[column_name].describe())

In [ ]:
breaks = pd.read_csv("data/raw/Water_Main_Breaks.csv", usecols=['X', 'Y', 'INCIDENT_DATE', 'BREAK_TYPE', 'BREAK_NATURE', 'BREAK_APPARENT_CAUSE', 
                                                                'BREAK_CATEGORIZATION', 'STREET', 'ASSETID', 'ASSET_EXISTS'])

['OBJECTID', 'WATMAINID', 'STATUS', 'PRESSURE_ZONE', 'ROADSEGMENTID',
       'MAP_LABEL', 'CATEGORY', 'PIPE_SIZE', 'MATERIAL', 'LINED', 'LINED_DATE',
       'LINED_MATERIAL', 'INSTALLATION_DATE', 'ACQUISITION', 'CONSULTANT',
       'OWNERSHIP', 'BRIDGE_MAIN', 'BRIDGE_DETAILS', 'CRITICALITY',
       'REL_CLEANING_AREA', 'REL_CLEANING_SUBAREA', 'UNDERSIZED',
       'SHALLOW_MAIN', 'CONDITION_SCORE', 'OVERSIZED', 'CLEANED', 'GlobalID',
       'Shape__Length']

In [ ]:
mains = pd.read_csv("data/raw/Water_Mains.csv", usecols=['OBJECTID', 'WATMAINID', 'PRESSURE_ZONE', 'ROADSEGMENTID',
                                                         'PIPE_SIZE', 'MATERIAL', 'INSTALLATION_DATE', 'CRITICALITY',
                                                         'CONDITION_SCORE'])

In [ ]:
# merge the two dataframes on the ASSETID column (which is the same as WATMAINID) and only keep the rows that have a match
merged_df = mains.merge(breaks, left_on='WATMAINID', right_on='ASSETID', how='inner')

In [ ]:
print(merged_df.shape)
merged_df.sample(3)

- move X and Y to front of frame
    - call them latitude and longitude
- lowercase the col names
- drop ID columns
- convert dates to datetime
- keep only assets that exist and then drop the column

In [ ]:
column_order = merged_df.columns.tolist()
column_order.insert(0, column_order.pop(column_order.index('X')))
column_order.insert(1, column_order.pop(column_order.index('Y')))
merged_df = merged_df.reindex(columns=column_order)

In [ ]:
merged_df.rename(columns={'X': 'latitude', 'Y': 'longitude'}, inplace=True)
merged_df.columns = merged_df.columns.str.lower()

In [ ]:
# drop ID columns
merged_df.drop(columns=['objectid', 'watmainid', 'assetid', 'roadsegmentid'], inplace=True)

# converting date columns to pd.to_datetime
merged_df['installation_date'] = pd.to_datetime(merged_df['installation_date'])
merged_df['incident_date'] = pd.to_datetime(merged_df['incident_date'])

# keep only assets that exist
merged_df = merged_df[merged_df['asset_exists'] == 'Y']

In [ ]:
print(merged_df.shape)
merged_df.sample(3)

In [ ]:
merged_df.drop(columns=['asset_exists'], inplace=True)

In [ ]:
merged_df.info()

In [ ]:
# drop rows with missing installation date
merged_df.dropna(subset=['installation_date'], inplace=True)

In [ ]:
merged_df.info()